In [2]:
import json

def extract_packages_from_spdx(spdx_file):
    with open(spdx_file, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    packages = data.get("packages", [])
    extracted_data = []
    
    for package in packages:
        name = package.get("name", "Unknown")
        version = package.get("versionInfo", "Unknown")
        extracted_data.append({"name": name, "version": version})
    
    return extracted_data

# 測試用：請將 'your_spdx.json' 替換成你的 SPDX JSON 檔案名稱
if __name__ == "__main__":
    spdx_file = "gpt2.json"  # 請填入你的 SPDX 檔案路徑
    packages = extract_packages_from_spdx(spdx_file)
    
    for package in packages:
        print(f"Package: {package['name']}, Version: {package['version']}")

Package: regex, Version: 2017.4.5
Package: requests, Version: 2.21.0
Package: tqdm, Version: 4.31.1
Package: ./gpt-2-master, Version: Unknown


In [1]:
import json

# 讀取 JSON 檔案
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

# 解析 JSON 資訊
def parse_model_info(data):
    model_info = {
        "Model Name": data.get("name", "Unknown"),
        "Version": data.get("model_version", "Unknown"),
        "Architecture": data.get("model_architecture", []),
        "Activation Function": data.get("activation_function", "Unknown"),
    }
    return model_info

# 測試
if __name__ == "__main__":
    file_path = "gpt2_info.json"  # 替換為你的 JSON 檔案路徑
    data = load_json(file_path)
    model_info = parse_model_info(data)
    for key, value in model_info.items():
        print(f"{key}: {value}")

Model Name: gpt2
Version: gpt2
Architecture: ['GPT2LMHeadModel']
Activation Function: gelu_new


In [2]:
#https://cve.mitre.org/cgi-bin/cvekey.cgi?keyword=Lunary
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
def cve_list(keyword):
    url = f'https://cve.mitre.org/cgi-bin/cvekey.cgi?keyword={keyword}'
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }

    # 發送請求
    response = requests.get(url, headers=headers)
    html = response.text

    # 解析 HTML
    soup = BeautifulSoup(html, 'html.parser')

    # 抓取 CVE ID
    component_cveid_list = [
        cve.text.strip()
        for cve in soup.select('table tr a[href^="https://www.cve.org/CVERecord?id="]')
    ]

    # 顯示結果
    # print(f'關鍵字 <{keyword}> 的 CVE 清單：')
    # print(component_cveid_list)
    return component_cveid_list

def cve_data(cve_id):
    # 設定 Chrome 選項
    options = Options()
    options.add_argument("--headless")  # 無頭模式
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # 啟動 WebDriver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    # 設定 CVE 網址
    url = f"https://www.cve.org/CVERecord?id={cve_id}"

    # 開啟網頁
    driver.get(url)

    # 等待 CVSS Score 元素出現（最多等 15 秒）
    try:
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'td[data-label="Score"]')))
    except:
        print("⚠️ CVE 資訊加載超時，可能無法獲取資料。")

    # 取得完整的 HTML
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 關閉瀏覽器
    driver.quit()

    # 爬取 CVE 資訊
    def safe_extract(soup, selector):
        element = soup.select_one(selector)
        return element.get_text(strip=True) if element else "N/A"

    Title = soup.select_one('span[style="text-transform: capitalize;"]').text.strip()
    Published = soup.select_one('span.has-text-weight-semibold + time').text.strip()
    Updated =  soup.select('span.has-text-weight-semibold + time')[1].text.strip()
    cvss_score = safe_extract(soup, 'td[data-label="Score"]')
    severity = safe_extract(soup, 'td[data-label="Severity"] span')
    cvss_version = safe_extract(soup, 'td[data-label="Version"]')
    vendor = soup.select_one('p.cve-product-status-heading + p').text.strip()
    product = soup.select('p.cve-product-status-heading + p')[1].text.strip()
    Default_Status = safe_extract(soup, 'span[class="has-text-weight-bold"] + text')
    affected_before = safe_extract(soup, 'span[class="cve-version"]')

    print(f"CVE ID: {cve_id}")
    print(f"Title: {Title}")
    print(f"Published: {Published}")
    print(f"Updated: {Updated}")
    print(f"CVSS Score: {cvss_score}")
    print(f"Severity: {severity}")
    print(f"CVSS Version: {cvss_version}")
    print(f"Vendor: {vendor}")
    print(f"product: {product}")
    print(f"Default_Status: {Default_Status}")
    print(f"affected_before: {affected_before}")


# for package in packages:
#     print(f"Package: {package['name']}, Version: {package['version']}")
#     keyword = package['name']
#     cve_lists = cve_list(keyword)
#     print(cve_lists)
#     if cve_lists:
#         cve_data(cve_lists[0])
    # for i in cve_lists:
    #     cve_data(i)
for key, value in model_info.items():
    print(f"{key}: {value}")
    keyword = value
    cve_lists = cve_list(keyword)
    print(cve_lists)
    if cve_lists:
        cve_data(cve_lists[0])

Model Name: gpt2
[]
Version: gpt2
[]
Architecture: ['GPT2LMHeadModel']
[]
Activation Function: gelu_new
[]


In [4]:
from transformers import AutoConfig, AutoModelForCausalLM

model_name = "deepseek-ai/deepseek-llm-7b-chat"  # 請用正確的 Hugging Face 名稱

try:
    config = AutoConfig.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    print(f"成功載入 {model_name}")
except Exception as e:
    print(f"載入 {model_name} 失敗: {str(e)}")


Loading checkpoint shards: 100%|██████████| 2/2 [01:45<00:00, 52.79s/it]


成功載入 deepseek-ai/deepseek-llm-7b-chat


In [3]:
!pip install transformers